# Ozon Counterfeit Late Fusion And Image Ablations

This notebook is a self-contained diagnostic pipeline for checking whether image embeddings improve the strong E1 tabular+text baseline. It does not modify the existing notebooks, `src`, or `scripts` files.

Main flow:
1. Load the same seller-aware train/val/test manifest.
2. Recompute E1 validation/test scores from the existing E1 artifact.
3. Train image-only and tabular+image side models.
4. Select the best image-side model on validation PR-AUC.
5. Fit late fusion alpha and threshold on validation only.
6. Report validation/test metrics for E1, image ablations, late fusion, and existing E2 artifacts.


In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

TRAIN_CSV = PROJECT_ROOT / 'data/ml_ozon_сounterfeit_data/ml_ozon_сounterfeit_train.csv'
TRAIN_IMAGES_DIR = PROJECT_ROOT / 'data/ml_ozon_сounterfeit_train_images'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
MANIFEST_PATH = ARTIFACTS_DIR / 'manifests/train_manifest.csv'
LATE_FUSION_REPORT_PATH = ARTIFACTS_DIR / 'reports/late_fusion_ablation_table.csv'

BASE_E1_EXPERIMENT = 'e1'
IMAGE_MODEL_NAME = 'efficientnet_b0'
# Alternative to try manually:
# IMAGE_MODEL_NAME = 'convnext_tiny.fb_in22k_ft_in1k'

IMAGE_SIDE_VARIANTS = ('image_only', 'tabular_image')
LATE_FUSION_ALPHAS = None  # Initialized after numpy import as np.linspace(0.0, 1.0, 21).

RUN_IMAGE_ABLATIONS = True
RUN_LATE_FUSION = True

# Use 1000 for smoke runs. Keep None for the full protocol.
MAX_SAMPLES_PER_SPLIT = None

DEVICE = 'mps'
BATCH_SIZE = 256
IMAGE_BATCH_SIZE = max(8, BATCH_SIZE // 4)
NUM_WORKERS = 0
LOGISTIC_MAX_ITER = 2000
LOGISTIC_C = 1.0

PROJECT_ROOT


PosixPath('/Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база')

In [2]:
if False:
    %pip install -r requirements.txt


In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ozon_counterfeit.constants import RANDOM_SEED
from ozon_counterfeit.data import load_train_frame, merge_manifest_with_frame
from ozon_counterfeit.encoders import FrozenImageEncoder, FrozenTextEncoder, load_cached_array, save_cached_array
from ozon_counterfeit.features import (
    TabularPreprocessor,
    build_modality_flags,
    build_text_corpus,
    extract_labels,
    infer_log_columns,
    infer_numeric_columns,
)
from ozon_counterfeit.models import EarlyFusionClassifier
from ozon_counterfeit.training import choose_threshold_by_f1, compute_metrics, load_pickle, save_pickle
from ozon_counterfeit.utils import ensure_dir, load_json, resolve_torch_device, save_json, set_seed, slugify

LATE_FUSION_ALPHAS = np.linspace(0.0, 1.0, 21)
set_seed(RANDOM_SEED)
print('Imports ready')


Imports ready


## Data And Split

The notebook uses the existing train manifest and preserves the same split protocol as the baseline notebooks.

In [4]:
def sample_per_split(frame: pd.DataFrame, limit: int) -> pd.DataFrame:
    sampled = []
    for _, split_frame in frame.groupby('split', sort=False):
        take = min(len(split_frame), limit)
        sampled.append(split_frame.sample(n=take, random_state=RANDOM_SEED))
    return pd.concat(sampled, axis=0).sort_values(['split', 'row_id']).reset_index(drop=True)


def load_split_frames(include_missing_images: bool = False) -> dict[str, pd.DataFrame]:
    train_frame = load_train_frame(TRAIN_CSV)
    manifest = pd.read_csv(MANIFEST_PATH)
    dataset = merge_manifest_with_frame(train_frame, manifest)

    if MAX_SAMPLES_PER_SPLIT:
        dataset = sample_per_split(dataset, MAX_SAMPLES_PER_SPLIT)

    if not include_missing_images:
        dataset = dataset.loc[dataset['has_image'] == 1].copy()

    split_frames = {
        split_name: split_frame.reset_index(drop=True)
        for split_name, split_frame in dataset.groupby('split', sort=False)
    }
    missing_splits = {'train', 'val', 'test'} - set(split_frames)
    if missing_splits:
        raise ValueError(f'Missing splits after filtering: {sorted(missing_splits)}')

    return split_frames


split_frames = load_split_frames(include_missing_images=False)
split_summary = pd.DataFrame(
    [
        {
            'split': split_name,
            'rows': len(split_frame),
            'positives': int(split_frame['resolution'].sum()),
            'positive_rate': float(split_frame['resolution'].mean()),
            'with_image_rows': int(split_frame['has_image'].sum()),
        }
        for split_name, split_frame in split_frames.items()
    ]
).sort_values('split')
display(split_summary)


,split,rows,positives,positive_rate,with_image_rows
2,test,32563,1957,0.060099,32563
0,train,132014,9097,0.068909,132014
1,val,31883,1903,0.059687,31883


## Shared Helpers

In [5]:
def experiment_slug(*parts: str) -> str:
    return '_'.join(slugify(str(part)) for part in parts if str(part))


IMAGE_MODEL_SLUG = slugify(IMAGE_MODEL_NAME)
BASE_OUTPUT_DIR = ARTIFACTS_DIR / 'experiments'
BASE_SCORES_DIR = BASE_OUTPUT_DIR / f'{BASE_E1_EXPERIMENT}_scores_{IMAGE_MODEL_SLUG}'


def load_cached_array_checked(cache_path: Path, expected_rows: int) -> np.ndarray | None:
    cached = load_cached_array(cache_path)
    if cached is None:
        return None
    if len(cached) != expected_rows:
        print(f'Ignoring stale cache with {len(cached)} rows, expected {expected_rows}: {cache_path}')
        return None
    return cached


def maybe_encode_text(split_name: str, split_frame: pd.DataFrame, cache_dir: Path, encoder: FrozenTextEncoder) -> np.ndarray:
    cache_path = cache_dir / f'text_{split_name}_{slugify(encoder.model_name)}.npy'
    cached = load_cached_array_checked(cache_path, len(split_frame))
    if cached is not None:
        return cached
    embeddings = encoder.encode(build_text_corpus(split_frame), batch_size=BATCH_SIZE)
    return save_cached_array(cache_path, embeddings)


def maybe_encode_images(split_name: str, split_frame: pd.DataFrame, cache_dir: Path, encoder: FrozenImageEncoder) -> np.ndarray:
    cache_path = cache_dir / f'image_{split_name}_{slugify(encoder.model_name)}.npy'
    cached = load_cached_array_checked(cache_path, len(split_frame))
    if cached is not None:
        return cached
    image_paths = split_frame['image_path'].astype(str).tolist()
    embeddings = encoder.encode(image_paths, batch_size=IMAGE_BATCH_SIZE, num_workers=NUM_WORKERS)
    return save_cached_array(cache_path, embeddings)


def labels_by_split(split_frames: dict[str, pd.DataFrame]) -> dict[str, np.ndarray]:
    return {split_name: extract_labels(split_frame) for split_name, split_frame in split_frames.items()}


def save_scores_npz(output_dir: Path, y_val: np.ndarray, y_test: np.ndarray, val_scores: np.ndarray, test_scores: np.ndarray) -> None:
    ensure_dir(output_dir)
    np.savez_compressed(
        output_dir / 'scores_val_test.npz',
        y_val=y_val.astype(np.float32),
        y_test=y_test.astype(np.float32),
        val_scores=val_scores.astype(np.float32),
        test_scores=test_scores.astype(np.float32),
    )


def load_scores_npz(output_dir: Path) -> dict[str, np.ndarray]:
    with np.load(output_dir / 'scores_val_test.npz') as payload:
        return {key: payload[key] for key in payload.files}


def metrics_from_scores(y_val: np.ndarray, y_test: np.ndarray, val_scores: np.ndarray, test_scores: np.ndarray) -> dict[str, object]:
    threshold = choose_threshold_by_f1(y_val, val_scores)
    return {
        'validation': compute_metrics(y_val, val_scores, threshold),
        'test': compute_metrics(y_test, test_scores, threshold),
    }


def save_experiment_payload(output_dir: Path, metadata: dict[str, object], metrics: dict[str, object]) -> None:
    ensure_dir(output_dir)
    save_json(output_dir / 'metadata.json', metadata)
    save_json(output_dir / 'metrics.json', metrics)


## Recompute E1 Scores

This cell loads the existing E1 model artifact and recomputes validation/test scores on the current split. It stores `scores_val_test.npz` so later threshold and late-fusion experiments do not depend only on aggregate metrics.

In [6]:
def recompute_e1_scores(split_frames: dict[str, pd.DataFrame]) -> dict[str, object]:
    e1_dir = BASE_OUTPUT_DIR / BASE_E1_EXPERIMENT
    if not (e1_dir / 'model.pt').exists():
        raise FileNotFoundError(f'Missing E1 model: {e1_dir / "model.pt"}. Run E1 in the original pipeline first.')
    if not (e1_dir / 'metadata.json').exists() or not (e1_dir / 'tabular_preprocessor.pkl').exists():
        raise FileNotFoundError(f'Missing E1 metadata/preprocessor in {e1_dir}. Run E1 in the original pipeline first.')

    output_dir = ensure_dir(BASE_SCORES_DIR)
    labels = labels_by_split(split_frames)
    metadata = load_json(e1_dir / 'metadata.json')
    preprocessor = load_pickle(e1_dir / 'tabular_preprocessor.pkl')
    device = resolve_torch_device(DEVICE)

    text_encoder = FrozenTextEncoder(str(metadata['text_model_name']), device=DEVICE)
    text_val = maybe_encode_text('val', split_frames['val'], output_dir / 'cache', text_encoder)
    text_test = maybe_encode_text('test', split_frames['test'], output_dir / 'cache', text_encoder)

    tab_val = preprocessor.transform(split_frames['val'])
    tab_test = preprocessor.transform(split_frames['test'])
    image_dim = int(metadata['image_embedding_dim'])
    image_val = np.zeros((len(split_frames['val']), image_dim), dtype=np.float32)
    image_test = np.zeros((len(split_frames['test']), image_dim), dtype=np.float32)
    val_flags = build_modality_flags(len(split_frames['val']), np.zeros(len(split_frames['val']), dtype=np.float32))
    test_flags = build_modality_flags(len(split_frames['test']), np.zeros(len(split_frames['test']), dtype=np.float32))

    model = EarlyFusionClassifier(
        tabular_dim=int(metadata['tabular_feature_dim']),
        text_dim=int(metadata['text_embedding_dim']),
        image_dim=image_dim,
        projection_dim=int(metadata['projection_dim']),
        hidden_dim=int(metadata['hidden_dim']),
        dropout=float(metadata['dropout']),
    )
    state_dict = torch.load(e1_dir / 'model.pt', map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    model.to(device)

    def predict(tabular: np.ndarray, text: np.ndarray, image: np.ndarray, flags: np.ndarray) -> np.ndarray:
        scores = []
        batch_size = BATCH_SIZE
        with torch.no_grad():
            for start in range(0, len(tabular), batch_size):
                stop = start + batch_size
                logits = model(
                    torch.from_numpy(tabular[start:stop].astype(np.float32)).to(device),
                    torch.from_numpy(text[start:stop].astype(np.float32)).to(device),
                    torch.from_numpy(image[start:stop].astype(np.float32)).to(device),
                    torch.from_numpy(flags[start:stop].astype(np.float32)).to(device),
                )
                scores.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(scores, axis=0).astype(np.float32)

    val_scores = predict(tab_val, text_val, image_val, val_flags)
    test_scores = predict(tab_test, text_test, image_test, test_flags)
    metrics = metrics_from_scores(labels['val'], labels['test'], val_scores, test_scores)
    metadata_out = {
        'experiment': f'{BASE_E1_EXPERIMENT}_recomputed_scores',
        'source_experiment': BASE_E1_EXPERIMENT,
        'text_model_name': metadata['text_model_name'],
        'image_model_name': metadata['image_model_name'],
        'max_samples_per_split': MAX_SAMPLES_PER_SPLIT,
        'train_rows': int(len(split_frames['train'])),
        'headline_eval_rows': int(len(split_frames['test'])),
    }
    save_scores_npz(output_dir, labels['val'], labels['test'], val_scores, test_scores)
    save_experiment_payload(output_dir, metadata_out, metrics)
    return {'output_dir': output_dir, 'metrics': metrics, 'metadata': metadata_out}


e1_scores_result = recompute_e1_scores(split_frames)
display(pd.DataFrame([e1_scores_result['metrics']['test']]).drop(columns=['classification_report', 'confusion_matrix']))
print(f"E1 scores saved to: {e1_scores_result['output_dir'] / 'scores_val_test.npz'}")


/Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/src/ozon_counterfeit/encoders.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embedding_dim = self.model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Batches:   0%|          | 0/128 [00:00<?, ?it/s]

/var/folders/76/0340_85s4pj0kpr1p4jfxxvw0000gn/T/ipykernel_47185/2930362396.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(e1_dir / 'model.pt',

,pr_auc,precision,recall,f1,threshold
0,0.593553,0.572175,0.615738,0.593158,0.49261


E1 scores saved to: /Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/artifacts/experiments/e1_scores_efficientnet-b0/scores_val_test.npz


## Image-Side Ablations

Train simple logistic-regression side models on frozen image embeddings. These models are intentionally simple: the goal is to test whether image embeddings contain useful signal and whether they can be blended with E1 without destabilizing it.

In [ ]:
def train_image_side_variant(variant: str, split_frames: dict[str, pd.DataFrame]) -> dict[str, object]:
    if variant not in IMAGE_SIDE_VARIANTS:
        raise ValueError(f'Unknown image-side variant: {variant}')

    output_dir = ensure_dir(BASE_OUTPUT_DIR / f'{variant}_{IMAGE_MODEL_SLUG}')
    cache_dir = ensure_dir(output_dir / 'cache')
    labels = labels_by_split(split_frames)

    image_encoder = FrozenImageEncoder(IMAGE_MODEL_NAME, device=DEVICE)
    image_train = maybe_encode_images('train', split_frames['train'], cache_dir, image_encoder)
    image_val = maybe_encode_images('val', split_frames['val'], cache_dir, image_encoder)
    image_test = maybe_encode_images('test', split_frames['test'], cache_dir, image_encoder)

    if variant == 'image_only':
        X_train = image_train
        X_val = image_val
        X_test = image_test
    elif variant == 'tabular_image':
        numeric_columns = infer_numeric_columns(load_train_frame(TRAIN_CSV))
        log_columns = infer_log_columns(numeric_columns)
        preprocessor = TabularPreprocessor(numeric_columns=numeric_columns, log_columns=log_columns)
        tab_train = preprocessor.fit_transform(split_frames['train'])
        tab_val = preprocessor.transform(split_frames['val'])
        tab_test = preprocessor.transform(split_frames['test'])
        X_train = np.hstack([tab_train, image_train]).astype(np.float32)
        X_val = np.hstack([tab_val, image_val]).astype(np.float32)
        X_test = np.hstack([tab_test, image_test]).astype(np.float32)
        save_pickle(output_dir / 'tabular_preprocessor.pkl', preprocessor)
    else:
        raise AssertionError(f'Unhandled variant: {variant}')

    model = make_pipeline(
        StandardScaler(with_mean=False if sparse.issparse(X_train) else True),
        LogisticRegression(
            max_iter=LOGISTIC_MAX_ITER,
            C=LOGISTIC_C,
            class_weight='balanced',
            solver='lbfgs',
            random_state=RANDOM_SEED,
        ),
    )
    model.fit(X_train, labels['train'].astype(int))
    val_scores = model.predict_proba(X_val)[:, 1].astype(np.float32)
    test_scores = model.predict_proba(X_test)[:, 1].astype(np.float32)
    metrics = metrics_from_scores(labels['val'], labels['test'], val_scores, test_scores)

    metadata = {
        'experiment': f'{variant}_{IMAGE_MODEL_SLUG}',
        'variant': variant,
        'model_family': 'logistic_regression',
        'image_model_name': IMAGE_MODEL_NAME,
        'image_embedding_dim': int(image_train.shape[1]),
        'max_samples_per_split': MAX_SAMPLES_PER_SPLIT,
        'train_rows': int(len(split_frames['train'])),
        'headline_eval_rows': int(len(split_frames['test'])),
        'logistic_max_iter': LOGISTIC_MAX_ITER,
        'logistic_c': LOGISTIC_C,
    }
    save_pickle(output_dir / 'model.pkl', model)
    save_scores_npz(output_dir, labels['val'], labels['test'], val_scores, test_scores)
    save_experiment_payload(output_dir, metadata, metrics)
    return {'variant': variant, 'output_dir': output_dir, 'metrics': metrics, 'metadata': metadata}


image_side_results = []
if RUN_IMAGE_ABLATIONS:
    for variant in IMAGE_SIDE_VARIANTS:
        result = train_image_side_variant(variant, split_frames)
        image_side_results.append(result)
        display(pd.DataFrame([{'variant': variant, **result['metrics']['test']}]).drop(columns=['classification_report', 'confusion_matrix']))
else:
    print('RUN_IMAGE_ABLATIONS is disabled')


Encoding images:   0%|          | 0/2063 [00:00<?, ?it/s]

Encoding images:   0%|          | 0/499 [00:00<?, ?it/s]

Encoding images:   0%|          | 0/509 [00:00<?, ?it/s]

,variant,pr_auc,precision,recall,f1,threshold
0,image_only,0.402667,0.464974,0.505365,0.484329,0.694914


Encoding images:   0%|          | 0/2063 [00:00<?, ?it/s]

## Late Fusion

Select the best image-side variant by validation PR-AUC, then tune `alpha` and threshold on validation only. Test metrics are computed once with the selected validation parameters.

In [ ]:
def select_best_image_side_result(image_side_results: list[dict[str, object]]) -> dict[str, object]:
    if not image_side_results:
        raise ValueError('No image-side results available. Enable RUN_IMAGE_ABLATIONS or load existing side scores first.')
    return max(image_side_results, key=lambda item: item['metrics']['validation']['pr_auc'])


def fit_late_fusion(e1_output_dir: Path, image_output_dir: Path, image_variant: str) -> dict[str, object]:
    e1_scores = load_scores_npz(e1_output_dir)
    image_scores = load_scores_npz(image_output_dir)

    y_val = e1_scores['y_val']
    y_test = e1_scores['y_test']
    if not np.array_equal(y_val, image_scores['y_val']) or not np.array_equal(y_test, image_scores['y_test']):
        raise ValueError('E1 and image-side scores are not aligned. Recompute both with the same split and MAX_SAMPLES_PER_SPLIT.')

    candidates = []
    for alpha in LATE_FUSION_ALPHAS:
        val_scores = alpha * e1_scores['val_scores'] + (1.0 - alpha) * image_scores['val_scores']
        threshold = choose_threshold_by_f1(y_val, val_scores)
        validation_metrics = compute_metrics(y_val, val_scores, threshold)
        candidates.append(
            {
                'alpha': float(alpha),
                'threshold': float(threshold),
                'validation_f1': validation_metrics['f1'],
                'validation_pr_auc': validation_metrics['pr_auc'],
            }
        )

    best = max(candidates, key=lambda item: (item['validation_f1'], item['validation_pr_auc']))
    val_scores = best['alpha'] * e1_scores['val_scores'] + (1.0 - best['alpha']) * image_scores['val_scores']
    test_scores = best['alpha'] * e1_scores['test_scores'] + (1.0 - best['alpha']) * image_scores['test_scores']
    metrics = {
        'validation': compute_metrics(y_val, val_scores, best['threshold']),
        'test': compute_metrics(y_test, test_scores, best['threshold']),
    }

    output_dir = ensure_dir(BASE_OUTPUT_DIR / f'late_fusion_{BASE_E1_EXPERIMENT}_{image_variant}_{IMAGE_MODEL_SLUG}')
    metadata = {
        'experiment': f'late_fusion_{BASE_E1_EXPERIMENT}_{image_variant}_{IMAGE_MODEL_SLUG}',
        'base_experiment': BASE_E1_EXPERIMENT,
        'image_side_variant': image_variant,
        'image_model_name': IMAGE_MODEL_NAME,
        'max_samples_per_split': MAX_SAMPLES_PER_SPLIT,
        'train_rows': int(len(split_frames['train'])),
        'headline_eval_rows': int(len(split_frames['test'])),
    }
    fusion_config = {
        'alpha': best['alpha'],
        'threshold': best['threshold'],
        'selected_by': 'validation_f1_then_validation_pr_auc',
        'candidates': candidates,
    }
    save_scores_npz(output_dir, y_val, y_test, val_scores, test_scores)
    save_experiment_payload(output_dir, metadata, metrics)
    save_json(output_dir / 'fusion_config.json', fusion_config)
    return {'output_dir': output_dir, 'metrics': metrics, 'metadata': metadata, 'fusion_config': fusion_config}


late_fusion_result = None
if RUN_LATE_FUSION:
    best_image_side = select_best_image_side_result(image_side_results)
    late_fusion_result = fit_late_fusion(
        e1_scores_result['output_dir'],
        best_image_side['output_dir'],
        best_image_side['variant'],
    )
    display(pd.DataFrame([late_fusion_result['metrics']['test']]).drop(columns=['classification_report', 'confusion_matrix']))
    print(f"Selected image-side variant: {best_image_side['variant']}")
    print(f"Late fusion config: {late_fusion_result['fusion_config']}")
else:
    print('RUN_LATE_FUSION is disabled')


## Report Table

In [ ]:
def row_from_metrics(experiment: str, metrics: dict[str, object], source: str) -> dict[str, object]:
    val = metrics['validation']
    test = metrics['test']
    return {
        'experiment': experiment,
        'source': source,
        'val_pr_auc': val['pr_auc'],
        'val_f1': val['f1'],
        'test_pr_auc': test['pr_auc'],
        'test_precision': test['precision'],
        'test_recall': test['recall'],
        'test_f1': test['f1'],
        'threshold': test['threshold'],
    }


rows = []
rows.append(row_from_metrics(e1_scores_result['metadata']['experiment'], e1_scores_result['metrics'], 'recomputed_scores'))
for result in image_side_results:
    rows.append(row_from_metrics(result['metadata']['experiment'], result['metrics'], 'image_side_ablation'))
if late_fusion_result is not None:
    rows.append(row_from_metrics(late_fusion_result['metadata']['experiment'], late_fusion_result['metrics'], 'late_fusion'))

for existing_experiment in ('e1', 'e2', 'e2_gated_convnext'):
    metrics_path = BASE_OUTPUT_DIR / existing_experiment / 'metrics.json'
    if metrics_path.exists():
        rows.append(row_from_metrics(existing_experiment, load_json(metrics_path), 'existing_artifact'))

report_table = pd.DataFrame(rows).sort_values(['source', 'experiment']).reset_index(drop=True)
ensure_dir(LATE_FUSION_REPORT_PATH.parent)
report_table.to_csv(LATE_FUSION_REPORT_PATH, index=False)
display(report_table)
print(f'Report saved to: {LATE_FUSION_REPORT_PATH}')
